# **Descriptive, Predictive, and Text Analytics for Airline Customer Reviews**

This is a classic end-to-end analytics + machine learning case (and a really strong one for your MSc Financial Analytics profile).

1. Problem Understanding

You are solving three layers of analytics:

1. Descriptive Analytics

→ What has happened?

Customer satisfaction trends
Ratings distribution
Differences across airlines, seat types, traveller types
2. Predictive Analytics

→ What will happen?

Predict ‘Recommended’ (Yes/No)
3. Text Analytics

→ Why did it happen?

Extract insights from review text


2. Data Preprocessing (Very Important Section)
Steps:
Handle missing values
Convert categorical variables
Process text data
Feature engineering

3. Descriptive Analytics
Key insights to extract:
Average rating by airline
Recommendation rate by:
Seat type
Traveller type
Correlation between service features


Insights you should write:
Business class → higher recommendation
Poor WiFi → negative sentiment
Cabin staff service → strong driver

4. Text Analytics (HIGH SCORING SECTION)
Step 1: Clean Text


Step 2: TF-IDF


Step 3: Sentiment Analysis


Step 4: Topic Modeling (Advanced)

5. Predictive Modeling
Target:

Recommended (0/1)



Models to use (compare performance):
1. Logistic Regression (baseline)

2. Random Forest (important)

3. XGBoost (high marks)

Train-Test Split:


6. Key Driver Analysis (VERY IMPORTANT)
Feature Importance:
Expected important factors:
Value for money ⭐
Cabin staff service ⭐
Seat comfort
Sentiment score
Overall rating

👉 This is your “business insight gold section


7. Model Improvement
Hyperparameter tuning (GridSearchCV)
Combine structured + text features
Try ensemble models


8. Business Recommendations

This is where you get distinction-level marks:

Example Insights:
Improve cabin staff training
Focus on value-for-money perception
Upgrade WiFi services
Target economy passengers (lowest satisfaction group)


**Import/install the necessary packages**

In [1]:
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

**Get the data from source destination**

In [2]:
# Use pd.read_excel() instead of pd.read_csv() for Excel files
df = pd.read_excel("/Users/apple/anaconda_projects/87bf154e-1ef3-462f-a28d-0318bdb5b72a/MGT0000_ITAG7105_new_dataset_final.xlsx")

# Target variable
df['Recommended'] = df['Recommended'].map({'yes':1, 'no':0})

# Convert Verified
df['Verified'] = df['Verified'].map({'Yes':1, 'No':0})

# Convert categorical - corrected column name "Type Of Traveller" with proper capitalization
cat_cols = ['Airline Name', 'Type Of Traveller', 'Seat Type', 'Route']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert ratings to numeric
rating_cols = ['Overall_Rating','Seat Comfort','Cabin Staff Service',
               'Food & Beverages','Inflight Entertainment',
               'Ground Service','Wifi & Connectivity','Value For Money']

for col in rating_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop missing target
df = df.dropna(subset=['Recommended'])

**Descriptive Analytics**

Key insights to extract:
Average rating by airline
Recommendation rate by:
Seat type
Traveller type
Correlation between service features


Insights you should write:
Business class → higher recommendation
Poor WiFi → negative sentiment
Cabin staff service → strong driver


In [3]:
print(df.describe())
print(df['Recommended'].value_counts(normalize=True))

       Overall_Rating  Verified  Seat Comfort  Cabin Staff Service  \
count     6104.000000       0.0   5598.000000          5540.000000   
mean         4.189875       NaN      2.587710             2.833755   
std          2.783961       NaN      1.469871             1.624081   
min          1.000000       NaN      1.000000             1.000000   
25%          2.000000       NaN      1.000000             1.000000   
50%          3.000000       NaN      2.000000             3.000000   
75%          7.000000       NaN      4.000000             5.000000   
max          9.000000       NaN      5.000000             5.000000   

       Food & Beverages  Inflight Entertainment  Ground Service  \
count       3961.000000             2648.000000     5905.000000   
mean           2.557183                2.198640        2.370195   
std            1.500109                1.432463        1.619806   
min            1.000000                1.000000        1.000000   
25%            1.000000           

**TEXT ANALYTICS**

In [4]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['clean_review'] = df['Review'].apply(clean_text)

**TF-IDF Features:**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300)
X_text = tfidf.fit_transform(df['clean_review'])

**Sentiment Score:**

In [6]:
from textblob import TextBlob

df['sentiment'] = df['clean_review'].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

In Report this could be written 
Positive sentiment → high recommendation
Negative sentiment → complaints (food, delays, rude staff)

**BUILD PREDICTION MODEL**

In [7]:
X = df[rating_cols + ['sentiment']]
y = df['Recommended']

**Train-Test:**

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Random Forest**

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

RandomForestClassifier()

**Evaluation**

In [10]:
from sklearn.metrics import classification_report, roc_auc_score

pred = model.predict(X_test)

print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       869
           1       0.96      0.94      0.95       367

    accuracy                           0.97      1236
   macro avg       0.97      0.96      0.96      1236
weighted avg       0.97      0.97      0.97      1236

ROC-AUC: 0.9955475146038384


**MOST IMPORTANT PART — KEY DRIVERS**

In [11]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

Value For Money           0.305854
Ground Service            0.247752
Cabin Staff Service       0.139996
sentiment                 0.132196
Seat Comfort              0.094715
Food & Beverages          0.031452
Overall_Rating            0.025921
Inflight Entertainment    0.013227
Wifi & Connectivity       0.008887
dtype: float64